In [6]:
%matplotlib inline

import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path().resolve().parent   
sys.path.insert(0, str(ROOT / "src"))

from modem.pipeline import run
from modem.blocks import (
    TextSource, RawBitSource, HexSource,
    BytesToBits, BitsToBytes, MessageSink,
    BPSKModulator, BPSKDemodulator, DBPSKDemodulator,
    QPSKModulator, QPSKDemodulator, DQPSKModulator, DQPSKDemodulator,
    BFSKModulator, BFSKDemodulator, DBFSKDemodulator,
    DummyModulator, DummyDemodulator,
    IQModulator, IQDemodulator,
    ZeroOrderHold, IntegrateAndDump,
    AWGNChannel, RayleighFlatFadingChannel, MultipathChannel, NoChannel, DopplerChannel, FrequencyCorrector,
    PrependBit, DifferentialEncoder,
    HammingCoder, HammingDecoder, DummyCoder, DummyDecoder,
    LMSLinearEqualizer
)

# Вспомогательные блоки (можно вынести в отдельный модуль, но пока так)
from modem import Block
class PrependArrayBlock(Block):
    def __init__(self, prepend, name="prepend"):
        super().__init__(name)
        self.prepend = np.asarray(prepend)
    def process(self, x, **kwargs):
        return np.concatenate([self.prepend, x])

class RemovePreambleBlock(Block):
    def __init__(self, n, name="remove"):
        super().__init__(name)
        self.n = n
    def process(self, x, **kwargs):
        return x[self.n:]

print("Импорты успешны")

Импорты успешны


In [7]:
import ipywidgets as widgets
from IPython.display import display

source_widget = widgets.Dropdown(
    options=[('Текст', 'text'), ('Биты', 'bits'), ('Hex', 'hex')],
    value='text',
    description='Источник:')

message_widget = widgets.Text(value='Hello, world!', description='Текст/биты:')

coding_widget = widgets.Dropdown(
    options=[('Хэмминг (7,4)', 'hamming'), ('Без кодирования', 'none')],
    value='hamming',
    description='Код:')

mod_widget = widgets.Dropdown(
    options=[('BPSK', 'bpsk'), ('DBPSK', 'dbpsk'),
             ('QPSK', 'qpsk'), ('DQPSK', 'dqpsk'),
             ('BFSK', 'bfsk'), ('DBFSK', 'dbfsk'),
             ('Без модуляции', 'none')],
    value='bpsk',
    description='Модуляция:')

channel_widget = widgets.Dropdown(
    options=[('AWGN', 'awgn'), ('Rayleigh + AWGN', 'rayleigh'),
             ('Многолучевой (с эквалайзером)', 'multipath'),('Без шума','none')],
    value='awgn',
    description='Канал:')

snr_widget = widgets.FloatSlider(value=20, min=-60, max=60, step=0.5, description='SNR (dB):')
sps_widget = widgets.IntSlider(value=20, min=1, max=200, step=1, description='Сэмплов/символ:')
fs_widget = widgets.FloatText(value=1000.0, description='Част. дискр.:')
fc_widget = widgets.FloatText(value=50.0, description='Несущая (Гц):')
use_carrier_widget = widgets.Checkbox(value=False, description='Перенос на несущую')

deviation_widget = widgets.FloatText(value=25.0, description='FSK девиация:')

block_size_widget = widgets.IntText(value=200, description='Размер блока:')
multipath_widget = widgets.Text(value='4,0.6', description='Лучи (задержка,коэф):')

freq_offset_widget = widgets.FloatSlider(
    value=0, min=0, max=5000, step=1,
    description='Допплер (Гц):',
    continuous_update=False
)
compensate_widget = widgets.Checkbox(value=False, description='Компенсировать допплер')

run_button = widgets.Button(description='Запустить моделирование', button_style='success')
block_view_widget = widgets.Dropdown(
    options=[
        'coder',
        'mod',
        'awgn',
        'demod',
        'decoder'
    ],
    value='mod',
    description='Блок:'
)

spectrum_widget = widgets.Dropdown(
    options=[
        ('Амплитудный', 'amplitude'),
        ('Фазовый', 'phase')
    ],
    value='amplitude',
    description='Спектр:'
)
# Группируем виджеты
out = widgets.Output()
ui = widgets.VBox([
    widgets.HBox([source_widget, message_widget]),
    widgets.HBox([coding_widget, mod_widget, channel_widget]),
    snr_widget, sps_widget,
    widgets.HBox([fs_widget, fc_widget, use_carrier_widget]),
    deviation_widget,
    block_size_widget, multipath_widget,
    freq_offset_widget, compensate_widget, 
    widgets.HBox([
        block_view_widget,
        spectrum_widget
    ]),
    run_button,
    out  # сюда будут падать результаты
])
display(ui)

In [8]:
def build_blocks_from_widgets():
    # Источник
    if source_widget.value == 'text':
        src = TextSource(name="source", message=message_widget.value)
    elif source_widget.value == 'bits':
        src = RawBitSource(name="source", bitstring=message_widget.value)
    else:
        src = HexSource(name="source", hexstring=message_widget.value)

    # Кодер/декодер
    if coding_widget.value == 'hamming':
        coder = HammingCoder(name="coder")
        decoder = HammingDecoder(name="decoder")
    else:
        coder = DummyCoder(name="coder")
        decoder = DummyDecoder(name="decoder")

    # Модулятор/демодулятор
    mode_map = {
        'bpsk':  ('BPSK',  BPSKModulator,  BPSKDemodulator,  False),
        'dbpsk': ('DBPSK', BPSKModulator,  DBPSKDemodulator, True),
        'qpsk':  ('QPSK',  QPSKModulator,  QPSKDemodulator,  False),
        'dqpsk': ('DQPSK', DQPSKModulator, DQPSKDemodulator, True),
        'bfsk':  ('BFSK',  BFSKModulator,  BFSKDemodulator,  False),
        'dbfsk': ('DBFSK', BFSKModulator,  DBFSKDemodulator, True),
        'none':  ('NONE',  DummyModulator,  DummyDemodulator, False),
         'gfsk':  ('GFSK',  GFSKModulator,  BFSKDemodulator,  False),   # демодулятор как у BFSK
    }
    mode_name, mod_class, demod_class, is_diff = mode_map[mod_widget.value]

    # Общие параметры
    sps = sps_widget.value
    fs = fs_widget.value
    snr_db = snr_widget.value
    fc = fc_widget.value
    use_carrier = use_carrier_widget.value and mode_name != 'NONE'
    deviation = deviation_widget.value if mode_name in ('BFSK', 'DBFSK') else 25.0

    ctx = {"samples_per_symbol": sps, "fs": fs}
    if use_carrier:
        ctx["fc"] = fc

    blocks = [src]
    if isinstance(src, (TextSource, HexSource)):
        blocks.append(BytesToBits(name="bytes_to_bits"))

    blocks.append(coder)

    if is_diff and mode_name in ("DBPSK", "DBFSK"):
        blocks += [PrependBit(name="prepend", bit=0),
                   DifferentialEncoder(name="diff_enc", initial_state=0)]

    channel = channel_widget.value
    multipath_bpsk = (channel == 'multipath' and mode_name == 'BPSK')

    # ---- Модулятор и ZOH (только если НЕ многолучевой BPSK) ----
    if not multipath_bpsk:
        if mode_name in ('BPSK', 'DBPSK', 'NONE'):
            blocks.append(mod_class(name="mod"))
        elif mode_name == 'QPSK':
            blocks.append(QPSKModulator(name="mod"))
        elif mode_name == 'DQPSK':
            blocks.append(DQPSKModulator(name="mod"))
        elif mode_name == 'BFSK':
            blocks.append(BFSKModulator(name="mod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))
        elif mode_name == 'DBFSK':
            blocks.append(BFSKModulator(name="mod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))
        elif mode_name == 'GFSK':
            # Здесь можно запросить BT через дополнительный виджет, а пока используем значение по умолчанию
            bt = 0.5   # можно сделать виджет bt_widget
            blocks.append(GFSKModulator(name="mod", bt=bt, deviation_hz=deviation,
                                    samples_per_symbol=sps, fs=fs))
        if mode_name in ('BPSK', 'DBPSK', 'QPSK', 'DQPSK', 'NONE'):
            blocks.append(ZeroOrderHold(name="zoh", samples_per_symbol=sps))

        if use_carrier:
            blocks.append(IQModulator(name="iq_mod", fc=fc))

    # ---- Канал ----
    if channel == 'none':
        blocks.append(NoChannel(name="no_channel"))
    elif channel == 'awgn':
        blocks.append(AWGNChannel(name="awgn", snr_db=snr_db, seed=np.random.randint(0,1000)))
    elif channel == 'rayleigh':
        blocks += [RayleighFlatFadingChannel(name="fading", seed=7, block_size=block_size_widget.value),
                   AWGNChannel(name="awgn", snr_db=snr_db, seed=np.random.randint(0,1000))]
    elif channel == 'multipath':
        if mode_name != 'BPSK':
            print("Многолучевой канал с эквалайзером только для BPSK. Переключено на AWGN.")
            blocks.append(AWGNChannel(name="awgn", snr_db=snr_db, seed=42))
        else:
            # ---- Специальная сборка многолучевого BPSK ----
            if use_carrier:
                print("Предупреждение: многолучевой режим с несущей не поддерживается, несущая отключена.")
                use_carrier = False
            # Парсим лучи
            taps_str = multipath_widget.value
            try:
                taps = []
                for pair in taps_str.split(';'):
                    parts = pair.strip().split(',')
                    d = int(parts[0])
                    c = float(parts[1].strip())
                    taps.append((d, c))
            except:
                taps = [(0, 1.0), (40, 0.6)]
            preamble = np.array([1,0,1,0,1,1,0,0,1,1,0,1,0,1,1,0,
                                 1,1,0,0,1,0,1,1,0,1,1,0,1,0,0,1], dtype=np.uint8)
            blocks += [
                PrependArrayBlock(prepend=preamble, name="prepend_preamble"),
                BPSKModulator(name="mod"),
                ZeroOrderHold(name="zoh", samples_per_symbol=sps),
                MultipathChannel(taps=taps, name="multipath"),
                AWGNChannel(name="awgn", snr_db=snr_db, seed=42),
                IntegrateAndDump(name="int_dump", samples_per_symbol=sps),
                LMSLinearEqualizer(name="eq", num_taps=11, step=0.1, preamble=None, dd=True),
                RemovePreambleBlock(n=len(preamble), name="remove_preamble"),
                BPSKDemodulator(name="demod"),
            ]
 # Допплеровский сдвиг и ручная компенсация
    freq_offset = freq_offset_widget.value
    if channel != 'none' and freq_offset != 0:
        blocks.append(DopplerChannel(name="doppler", freq_offset_hz=freq_offset))
        if compensate_widget.value:
            blocks.append(FrequencyCorrector(name="freq_corr", freq_offset_hz=freq_offset))
    # ---- IQ демодулятор (если включён и не многолучевой BPSK) ----
    if use_carrier and not multipath_bpsk:
        blocks.append(IQDemodulator(name="iq_demod", fc=fc))

    # ---- Демодулятор / обработка (если ещё не добавлен) ----
    if not multipath_bpsk:
        if mode_name in ('BPSK', 'DBPSK', 'QPSK', 'DQPSK', 'NONE'):
            blocks.append(IntegrateAndDump(name="int_dump", samples_per_symbol=sps))
            if mode_name == 'BPSK':
                blocks.append(BPSKDemodulator(name="demod"))
            elif mode_name == 'DBPSK':
                blocks.append(DBPSKDemodulator(name="demod"))
            elif mode_name == 'QPSK':
                blocks.append(QPSKDemodulator(name="demod"))
            elif mode_name == 'DQPSK':
                blocks.append(DQPSKDemodulator(name="demod"))
            elif mode_name == 'NONE':
                blocks.append(DummyDemodulator(name="demod"))
        elif mode_name in ('BFSK', 'DBFSK'):
            if mode_name == 'BFSK':
                blocks.append(BFSKDemodulator(name="demod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))
            else:
                blocks.append(DBFSKDemodulator(name="demod", deviation_hz=deviation, samples_per_symbol=sps, fs=fs))

    # ---- Декодер ----
    blocks.append(decoder)

    # ---- Восстановление сообщения ----
    if not isinstance(src, RawBitSource):
        blocks += [BitsToBytes(name="bits_to_bytes"),
                   MessageSink(name="msg_sink", errors="replace")]

    return blocks, ctx

In [9]:
def visualize_signal(signal, fs=1.0, mode="amplitude", title=""):
    signal = np.asarray(signal)

    n = len(signal)

    if n == 0:
        print("Пустой сигнал")
        return

    t = np.arange(n) / fs

    spectrum = np.fft.fftshift(np.fft.fft(signal))
    freqs = np.fft.fftshift(np.fft.fftfreq(n, d=1/fs))

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))

    # ====================================
    # СЛЕВА — СПЕКТР
    # ====================================

    if mode == "amplitude":
        ax[0].plot(freqs, np.abs(spectrum))
        ax[0].set_title("Amplitude Spectrum")

    elif mode == "phase":
        ax[0].plot(freqs, np.unwrap(np.angle(spectrum)))
        ax[0].set_title("Phase Spectrum")

    ax[0].set_xlabel("Frequency [Hz]")
    ax[0].grid(True)

    # ====================================
    # СПРАВА — ВРЕМЕННАЯ ОБЛАСТЬ
    # ====================================

    if np.iscomplexobj(signal):
        ax[1].plot(t, np.real(signal), label="I")
        ax[1].plot(t, np.imag(signal), label="Q")
        ax[1].legend()
    else:
        ax[1].plot(t, signal)

    ax[1].set_title("Time Domain")
    ax[1].set_xlabel("Time [s]")
    ax[1].grid(True)

    fig.suptitle(title)

    plt.tight_layout()
    plt.show()

In [10]:
# Ячейка: функция запуска и привязка кнопки (выполняйте после создания всех виджетов и out)

def run_simulation(b):
    out.clear_output(wait=True)
    with out:
        blocks, ctx = build_blocks_from_widgets()
        # Если есть эквалайзер, добавим преамбулу
        for blk in blocks:
            if isinstance(blk, LMSLinearEqualizer):
                preamble_bits = np.array([1,0,1,0,1,1,0,0,1,1,0,1,0,1,1,0,
                                          1,1,0,0,1,0,1,1,0,1,1,0,1,0,0,1], dtype=np.uint8)
                mod = BPSKModulator()
                preamble_syms = mod(preamble_bits)
                ctx["preamble"] = preamble_syms
                break

        capture = {}
        out_val = run(blocks, x=None, ctx=ctx, capture=capture)
        
        src = blocks[0]
        print('=== РЕЗУЛЬТАТЫ ===')
        if hasattr(src, 'message'):
            print(f"Передано: {src.message}")
        elif hasattr(src, 'bitstring'):
            print(f"Передано бит: {src.bitstring}")
        elif hasattr(src, 'hexstring'):
            print(f"Передано hex: {src.hexstring}")

        try:
            tx_bits = capture.get("bytes_to_bits", None)
            if tx_bits is None and hasattr(src, 'bitstring'):
                tx_bits = np.array([int(b) for b in src.bitstring], dtype=np.uint8)
        except:
            tx_bits = None

        decoded = None
        for blk in blocks:
            if isinstance(blk, (HammingDecoder, DummyDecoder)):
                decoded = capture.get(blk.name, None)
                break
        if tx_bits is not None and decoded is not None:
            m = min(len(tx_bits), len(decoded))
            errs = np.sum(tx_bits[:m] != decoded[:m])
            ber = errs/m if m else 0
            print(f"Битовые ошибки: {errs}/{m} (BER={ber:.2e})")
       
        if isinstance(src, TextSource) and isinstance(out_val, str):
            print(f"Принято: {out_val!r}")
        elif isinstance(src, RawBitSource) and decoded is not None:
            recovered_bits = decoded[:len(src.bitstring)]
            print(f"Принятые биты: {''.join(str(b) for b in recovered_bits)}")
        elif isinstance(src, HexSource) and isinstance(out_val, bytes):
            print(f"Принятые hex: {out_val.hex()}")
        # ====================================
        # ВИЗУАЛИЗАЦИЯ
        # ====================================

        selected_block = block_view_widget.value

        if selected_block not in capture:
            print(f"Блок '{selected_block}' не найден")
            print("Доступные блоки:", list(capture.keys()))
        else:
            visualize_signal(
                capture[selected_block],
                fs=ctx["fs"],
                mode=spectrum_widget.value,
                title=selected_block
            )
        # Графики
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
        if "zoh" in capture:
            ax1.plot(capture["zoh"][:200].real, label='TX (I)')
        elif "mod" in capture:
            ax1.plot(capture["mod"][:200].real, label='TX (I)', alpha=0.7)
            ax1.plot(capture["mod"][:200].imag, label='TX (Q)', alpha=0.5)
        ax1.set_title("Переданный сигнал")
        ax1.grid(True); ax1.legend()

        rx_sig = None
        for key in ("awgn", "multipath", "no_channel"):
            if key in capture:
                rx_sig = capture[key]
                break
        # Если ничего не нашли, но есть другие блоки – можно взять выход последнего блока
        if rx_sig is None:
            rx_sig = out   # out – результат run (выход последнего блока), может быть что угодно, но для графика сойдёт

        if rx_sig is not None:
            ax2.plot(rx_sig[:200].real, label='RX (I)', color='r')
            if np.iscomplexobj(rx_sig) and np.any(np.abs(rx_sig.imag) > 1e-12):
                ax2.plot(rx_sig[:200].imag, label='RX (Q)', color='m', alpha=0.5)
            ax2.set_title("Принятый сигнал")
            ax2.grid(True); ax2.legend()

        plt.tight_layout()
        plt.show()

run_button.on_click(run_simulation)